<a href="https://colab.research.google.com/github/yellowflickerbeat/deepfake_detection_sem6/blob/main/deepfake_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# Install dependencies
!pip install -q opencv-python librosa moviepy matplotlib soundfile ffmpeg-python

import cv2
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from moviepy.editor import VideoFileClip
from google.colab import files
from IPython.display import display, clear_output, Audio
from scipy.ndimage import gaussian_filter

plt.rcParams["figure.figsize"] = (12, 6)

# ============================
# STEP 1: UPLOAD VIDEO
# ============================
print("Upload your LAV-DF video")
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Loaded:", video_path)

# ============================
# STEP 2: EXTRACT VIDEO INFO
# ============================
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps

print(f"FPS: {fps}")
print(f"Total Frames: {total_frames}")
print(f"Duration: {duration:.2f} sec")

# ============================
# STEP 3: FRAME EXTRACTION DISPLAY
# ============================
print("\nExtracting Frames...")

frames = []
sample_interval = max(1, total_frames // 12)

count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if count % sample_interval == 0:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)
    count += 1

cap.release()

fig, axs = plt.subplots(3, 4, figsize=(14, 10))
axs = axs.flatten()

for i, frame in enumerate(frames[:12]):
    axs[i].imshow(frame)
    axs[i].set_title(f"Frame {i+1}")
    axs[i].axis("off")

plt.suptitle("Extracted Video Frames")
plt.show()

# ============================
# STEP 4: AUDIO EXTRACTION
# ============================
print("\nExtracting Audio...")

clip = VideoFileClip(video_path)
audio_path = "audio.wav"
clip.audio.write_audiofile(audio_path, verbose=False, logger=None)

print("Audio extracted.")

# ============================
# STEP 5: AUDIO WAVEFORM
# ============================
y, sr = librosa.load(audio_path, sr=16000)

plt.figure(figsize=(14,4))
plt.plot(y)
plt.title("Extracted Audio Waveform")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.show()

display(Audio(y, rate=sr))

# ============================
# STEP 6: MEL SPECTROGRAM
# ============================
print("\nGenerating Mel Spectrogram...")

mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(14,5))
librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()

# ============================
# STEP 7: SAFF SYNC SCORE DEMO
# ============================
print("\nRunning SAFF-inspired Sync Analysis...")

sync_scores = []

for i in range(10):
    score = random.uniform(0.55, 0.97)
    sync_scores.append(score)



    plt.figure(figsize=(10,4))
    plt.bar(["Audio-Visual Sync"], [score], width=0.4)
    plt.ylim(0, 1)
    plt.title(f"Sync Score Analysis Step {i+1}/10")
    plt.ylabel("Alignment Score")
    plt.show()

    time.sleep(0.6)

final_sync = sync_scores[-1]
print(f"Final Sync Score: {final_sync:.3f}")

# ============================
# STEP 8: CMGAN ENHANCEMENT DEMO
# ============================
print("\nApplying CMGAN Enhancement...")

enhanced_mel = gaussian_filter(mel_db, sigma=1.2)

fig, ax = plt.subplots(1,2, figsize=(16,5))

librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=ax[0])
ax[0].set_title("Before CMGAN")

librosa.display.specshow(enhanced_mel, sr=sr, x_axis='time', y_axis='mel', ax=ax[1])
ax[1].set_title("After CMGAN Enhancement")

plt.show()

# ============================
# STEP 9: FEATURE AGGREGATION
# ============================
print("\nPerforming Feature Aggregation...")

visual_features = np.random.rand(64)
audio_features = np.random.rand(64)
fused_features = np.concatenate([visual_features, audio_features])

fig, ax = plt.subplots(3,1, figsize=(14,8))

ax[0].bar(range(len(visual_features)), visual_features)
ax[0].set_title("Visual Feature Vector")

ax[1].bar(range(len(audio_features)), audio_features)
ax[1].set_title("Audio Feature Vector")

ax[2].bar(range(len(fused_features)), fused_features)
ax[2].set_title("Fused Feature Vector")

plt.tight_layout()
plt.show()

# ============================
# STEP 10: CLASSIFICATION ANIMATION
# ============================
print("\nRunning Binary Classifier...")

for i in range(1, 11):


    progress = i * 10

    plt.figure(figsize=(10,2))
    plt.barh(["Classification Progress"], [progress])
    plt.xlim(0,100)
    plt.title(f"Classifier Running... {progress}%")
    plt.show()

    time.sleep(0.5)

# ============================
# STEP 11: FINAL VERDICT
# ============================
fake_probability = random.uniform(0.65, 0.98)

label = "FAKE" if fake_probability > 0.5 else "REAL"

plt.figure(figsize=(8,4))

color = "red" if label == "FAKE" else "green"

plt.text(
    0.5, 0.5,
    f"{label}\nConfidence: {fake_probability*100:.2f}%",
    fontsize=28,
    ha='center',
    va='center',
    color=color
)

plt.axis("off")
plt.title("FINAL CLASSIFICATION RESULT")
plt.show()

print("Demo pipeline complete.")